In [1]:
%load_ext autoreload
%autoreload 2

import sys
import torch
from pathlib import Path
import urllib.request

# Aponta para a pasta onde estão seus códigos estruturados
sys.path.append(str(Path.cwd().parent / "Codigo"))

# Importando o que você já fez (Capítulos anteriores)
from bpe import BPETokenizer
from dataset import create_dataloader_v1
from model import GPTModel 

# Importando as funções do Entregável 2
from avaliacao import calc_loss_batch, evaluate_metrics

print("Módulos carregados. Pronto para avaliação quantitativa.")

Módulos carregados. Pronto para avaliação quantitativa.


In [2]:
# Download de um texto pequeno para teste
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
urllib.request.urlretrieve(url, "the-verdict.txt")

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    texto = f.read()

# Dividindo dados para ter conjuntos de Treino e Validação distintos
split_idx = int(0.9 * len(texto))
train_data = texto[:split_idx]
val_data = texto[split_idx:]

# Configuração que seu GPTModel espera
bpe = BPETokenizer("gpt2")
GPT_CONFIG = {
    "vocab_size": bpe.vocab_size, "context_length": 256, "emb_dim": 768,
    "n_heads": 12, "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False
}

# Usando sua função de dataloader
train_loader = create_dataloader_v1(train_data, batch_size=2, max_length=256, stride=256, drop_last=True)
val_loader = create_dataloader_v1(val_data, batch_size=2, max_length=256, stride=256, drop_last=False)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(123)

# Instancia o modelo criado por você no Capítulo 4
model = GPTModel(GPT_CONFIG).to(device)

print("Avaliando o estado inicial (pesos aleatórios)...")
train_l, val_l, train_p, val_p = evaluate_metrics(model, train_loader, val_loader, device)

print(f"Estado Inicial -> Train Loss: {train_l:.4f} | Train Perplexity: {train_p:.2f}")
print(f"Estado Inicial -> Val Loss:   {val_l:.4f} | Val Perplexity:   {val_p:.2f}")

Avaliando o estado inicial (pesos aleatórios)...
Estado Inicial -> Train Loss: 10.9876 | Train Perplexity: 59135.23
Estado Inicial -> Val Loss:   10.9811 | Val Perplexity:   58753.49


In [4]:
# Para cumprir o requisito de "avaliar a evolução", rodamos alguns passos de otimização
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
model.train()

print("Rodando 10 iterações de treino para observar a evolução...")
for i, (input_batch, target_batch) in enumerate(train_loader):
    if i >= 10: break
    
    optimizer.zero_grad()
    loss = calc_loss_batch(input_batch, target_batch, model, device)
    loss.backward()
    optimizer.step()
    
print("Iterações concluídas.")

Rodando 10 iterações de treino para observar a evolução...
Iterações concluídas.


In [5]:
print("Avaliando o modelo após 10 passos de treinamento...")
train_l_evol, val_l_evol, train_p_evol, val_p_evol = evaluate_metrics(model, train_loader, val_loader, device)

print(f"Estado Evoluído -> Train Loss: {train_l_evol:.4f} | Train Perplexity: {train_p_evol:.2f}")
print(f"Estado Evoluído -> Val Loss:   {val_l_evol:.4f} | Val Perplexity:   {val_p_evol:.2f}")

Avaliando o modelo após 10 passos de treinamento...
Estado Evoluído -> Train Loss: 6.8387 | Train Perplexity: 933.24
Estado Evoluído -> Val Loss:   7.1185 | Val Perplexity:   1234.59
